In [2]:
import pandas as pd
from datetime import datetime

In [40]:
hiv_patient_tracker=pd.read_json('hiv_patient_tracker_0_20240716163845_decrypted.json')

In [41]:
hiv_patient_tracker.head()

,biometric_status,cause_of_death,person_uuid,last_modified_date,status_uuid,uuid,archived,date_of_death,facility_id,reason_for_discountinuation,...,reason_for_tracking,care_in_facility_discountinued,reason_for_loss_to_follow_up,reason_for_tracking_others,referred_for,created_date,reason_for_loss_to_follow_up_others,dsd_model,date_of_discontinuation,referred_for_others
0,,,c1d250a5-ae05-467a-8d0c-7feaabdf4545,2024-07-09 14:34:10.253,1afea416-de97-476d-897a-1a16182e508d,6fdebcfe-a710-455f-8953-b4bd22ec074f,0,,2005,Self-transfer to another facility,...,REASON_TRACKING_MISSED_APPOINTMENT,Yes,,,,2024-07-09 14:34:09.64,,FBM,2024-07-06,
1,,,c18d183a-f7ae-4724-8367-92d9495a07e0,2024-07-09 14:43:41.209,91531332-9a06-4dab-808e-edad1bd61426,cb441b04-ddd1-4f38-b611-8e509292d35d,0,,2005,Self-transfer to another facility,...,REASON_TRACKING_MISSED_APPOINTMENT,Yes,,,,2024-07-09 14:43:40.583,,FBM,2024-07-06,
2,,,452b9e2e-a300-4f60-8987-eb0f17e41115,2024-07-09 14:58:52.04,c146978f-d633-4290-9d6a-f3e0da648458,9da86737-b166-4d1a-8632-e318beb19df8,0,,2005,Self-transfer to another facility,...,REASON_TRACKING_MISSED_APPOINTMENT,Yes,,,,2024-07-09 14:58:51.523,,,2024-07-06,
3,,,ea014f0c-3a2a-4b8a-9509-470c007f4200,2024-07-09 15:08:52.101,89ef203d-0120-4219-a48d-f70b855bf67f,ea3bc83e-c0ed-4aa2-8c29-732a6bdc8bf0,0,,2005,Self-transfer to another facility,...,REASON_TRACKING_MISSED_APPOINTMENT,Yes,,,,2024-07-09 15:08:51.894,,FBM,2024-07-06,
4,,,c6affbf3-98b0-46dd-bb8d-6c3518746f65,2024-07-09 15:14:35.78,7ebfcbb7-a670-49f1-a987-72af4831fb3b,12365fee-5ba3-4499-ab90-993c61d10960,0,,2005,Self-transfer to another facility,...,REASON_TRACKING_MISSED_APPOINTMENT,Yes,,,,2024-07-09 15:12:23.39,,,2024-07-08,


In [56]:
def _date_validation(df):   
    date_columns = [col for col in df.columns if col.startswith('date') or col.endswith('date')]
    problematic_dates = {}
    if date_columns:
        df_dates = df[date_columns].fillna('2024-01-01')
        validity_results = {}
        for col in date_columns:
            try:
                pd.to_datetime(df_dates[col])
                validity_results[col] = True  # Date is valid
            except ValueError:
                validity_results[col] = False  # Date is invalid or column contains non-date data
        failed_columns = [key for key, value in validity_results.items() if value is False]
        if failed_columns != []:
            df_with_bad_date_columns=df[failed_columns]
            for column in failed_columns:
                for idx, value in enumerate(df_with_bad_date_columns[column]):
                    try:
                        pd.to_datetime(value, errors='raise')
                    except (TypeError, ValueError):
                        if column not in problematic_dates:
                            column = 'column_name: ' + column
                            problematic_dates[column] = []
                        problematic_dates[column].append((f'record {idx+1}, value => {value}'))
        if problematic_dates != {}:
            return problematic_dates
        else:
            return {}
    else:
        return {}

validation_result=_date_validation(hiv_patient_tracker)
validation_result

{'column_name: date_last_appointment': ['record 3, value => 0003-11-09']}